# IL3.2: Análisis de Trazabilidad y Logs
## Notebook 3: Logs estructurados en JSON para análisis automático

### Objetivo:
Aprender a pasar de logs en texto plano a logs estructurados en formato JSON Lines (`.jsonl`). Esto facilita el análisis automatizado utilizando herramientas de ciencia de datos en Python, permitiendo calcular métricas críticas de rendimiento y costos.

### ¿Por qué Logs Estructurados?
Los logs de texto plano son excelentes para lectura humana, pero son difíciles de procesar automáticamente en lote. Al estructurar cada entrada como un objeto JSON:
- Facilitamos el análisis con pandas o bases de datos como Elasticsearch.
- Aseguramos que campos importantes (latencia, tokens, éxito, errores) estén claramente tipados.
- Cada línea del archivo de log contiene un único objeto JSON (formato JSON Lines o `.jsonl`).


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
!pip install pandas langchain langchain-openai langchain_classic wikipedia LangSmith

In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


c:\Users\realm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM de LangChain configurado.
✅ Agente y herramientas listos.


### Estructuración de Logs de un Agente LLM Real
Implementaremos una función envoltoria (`run_wikipedia_agent_structured`) que ejecuta el agente real de Wikipedia (`agent_executor`) y registra una entrada JSON detallando el `trace_id`, `user_id`, la consulta original, la respuesta final, la latencia del proceso, si se usó la herramienta externa y si la ejecución fue exitosa.


In [2]:
import json
import time
import uuid
import random
from datetime import datetime
from langchain_core.callbacks import BaseCallbackHandler

log_file = "agent_structured.jsonl"

def write_log(entry):
    # Escribir la entrada como una línea en formato JSON
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def run_wikipedia_agent_structured(user_id, query):
    trace_id = str(uuid.uuid4())
    start_time = time.time()
    
    success = True
    error_type = None
    tool_used = None
    
    # Callback simple de LangChain para detectar si el agente decide usar la herramienta de Wikipedia
    class ToolTrackerHandler(BaseCallbackHandler):
        def on_tool_start(self, serialized, input_str, **kwargs):
            nonlocal tool_used
            tool_used = serialized.get("name", "get_wikipedia_summary")
            
    tracker = ToolTrackerHandler()
    
    try:
        if llm is None:
            # Simulación por ausencia de API key
            time.sleep(random.uniform(0.1, 0.3))
            if "error" in query.lower():
                raise ValueError("Error simulado de conexión con el modelo.")
            tool_used = "get_wikipedia_summary" if "buscar" in query.lower() or "quién" in query.lower() else None
            response_text = f"Respuesta simulada sobre {query}."
        else:
            response = agent_executor.invoke(
                {"input": query},
                config={"callbacks": [tracker]}
            )
            response_text = response.get("output", "")
    except Exception as e:
        success = False
        error_type = type(e).__name__
        response_text = "No fue posible completar la solicitud debido a un error interno."
        
    duration = round(time.time() - start_time, 4)
    # Estimación de tokens usados basados en caracteres (aproximación didáctica)
    tokens_est = (len(query) + len(response_text)) // 4
    
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "trace_id": trace_id,
        "user_id": user_id,
        "query": query,
        "response": response_text,
        "response_time": duration,
        "tokens_used": tokens_est,
        "success": success,
        "tool_used": tool_used,
        "error_type": error_type,
        "agent_version": "2.0-langchain"
    }
    
    write_log(log_entry)
    return response_text


### Simulación de Múltiples Interacciones Reales
Ejecutaremos consultas desde diferentes IDs de usuarios ficticios para poblar el archivo JSONL con registros válidos.


In [4]:
# Inicializar archivo de logs limpio
with open(log_file, "w", encoding="utf-8") as f:
    pass

queries = [
    ("user_1", "Explícame qué es trazabilidad"),
    ("user_2", "¿Quién fue Marie Curie?"),
    ("user_1", "provoca un error de ejecución en la API"),
    ("user_3", "Dame consejos de estudio rápido"),
    ("user_2", "¿Dónde queda la Patagonia?"),
    ("user_4", "Hola, buenas tardes"),
    ("user_1", "error crítico de red"),
    ("user_3", "Busca datos sobre el planeta Marte")
]

print("Ejecutando consultas y generando logs estructurados...")
for user_id, query in queries:
    run_wikipedia_agent_structured(user_id, query)
print("Logs guardados exitosamente en 'agent_structured.jsonl'.")


Ejecutando consultas y generando logs estructurados...


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'trazabilidad'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)La trazabilidad es el proceso mediante el cual se puede rastrear el recorrido de un producto, servicio o información a lo largo de toda su cadena de suministro, desde su origen hasta su destino final. Este concepto es fundamental en sectores como la industria alimentaria, farmacéutica y logística, ya que permite garantizar la calidad, seguridad y cumplimiento de normativas, además de facilitar la identificación de problemas en caso de incidentes.

> Finished chain.


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Marie Curie'}`


Maria Salomea Skłodowska-Curie,​​ más conocida como Marie Curie​​ o Madame Curie (Varsovia, 7 de noviembre de 1867-Passy, 4 de julio de 1934), fue una física y química polaca, luego naturalizada fr

### Análisis de Logs con Pandas
Ahora que los logs están estructurados en formato JSONL, cargarlos en un DataFrame de `pandas` y realizar consultas analíticas es sumamente sencillo y no requiere regex complejas.


In [5]:
import pandas as pd

rows = []
with open("agent_structured.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("--- Estructura del DataFrame de Logs ---")
print(df.head())

print("\n--- Estadísticas del Agente de Wikipedia ---")
print("Total interacciones:", len(df))
print(f"Tasa de éxito: {df['success'].mean() * 100:.2f}%")
print(f"Tiempo promedio de respuesta: {df['response_time'].mean():.4f} segundos")
print(f"Consumo estimado de tokens: {df['tokens_used'].mean():.1f} tokens")
print("\n--- Llamadas a Herramientas de Wikipedia ---")
print(df["tool_used"].value_counts(dropna=False))


--- Estructura del DataFrame de Logs ---
                    timestamp                              trace_id user_id  \
0  2026-06-05T19:11:00.378167  446ca136-09b4-4335-a623-f62877755f23  user_1   
1  2026-06-05T19:11:07.594818  669037f7-2fa4-4b12-baf1-8181c91659d3  user_2   
2  2026-06-05T19:11:10.147973  edfad0dc-8e9c-4099-a9c0-2d1fd178de90  user_1   
3  2026-06-05T19:11:14.497909  7f00f6af-9776-4099-b3cf-ac23fa2412b0  user_3   
4  2026-06-05T19:11:16.923078  951ec5cb-8d4a-4f15-ba17-78c9f13f3d82  user_2   

                                     query  \
0            Explícame qué es trazabilidad   
1                  ¿Quién fue Marie Curie?   
2  provoca un error de ejecución en la API   
3          Dame consejos de estudio rápido   
4               ¿Dónde queda la Patagonia?   

                                            response  response_time  \
0  La trazabilidad es el proceso mediante el cual...         5.7753   
1  Marie Curie fue una física y química polaca, n...         7.21

### 🛠️ Reto Práctico (Mini-entrega)

**Instrucciones:**
1. Modifica la función `run_wikipedia_agent_structured` para incluir un campo adicional en la entrada de log llamado `model_name` (puedes extraerlo del objeto `llm.model_name` o usar el string `"gpt-4o"`).
2. Ejecuta 5 nuevas consultas al agente real de Wikipedia utilizando diferentes usuarios.
3. Carga el archivo `.jsonl` en un DataFrame de Pandas y genera una tabla resumen que muestre por cada `user_id`:
   - El total de consultas.
   - El promedio de latencia (`response_time`).
   - El porcentaje de éxito (`success`).


In [ ]:
# Desarrolla tu solución aquí

# 1. Función modificada con model_name

# 2. Invocaciones de prueba

# 3. Carga en Pandas y tabla resumen por usuario


### 📝 Preguntas de Análisis
1. **¿Qué ventajas operativas ofrece el formato JSON Lines (`.jsonl`) sobre un archivo JSON estándar que contiene una lista (ej. `[...]`) al registrar logs de un servidor en tiempo real?**
2. **Si necesitas calcular el costo financiero exacto del uso de tu agente en producción, ¿qué campos específicos adicionales deberías registrar en el JSON de logs?**
3. **¿Cómo facilita un log estructurado la creación de alertas en tiempo real (por ejemplo, si el tiempo de respuesta promedio supera los 5 segundos)?**
